<a href="https://colab.research.google.com/github/kakopappa/On-policy-distillation-Sharp-CV-P09FX-AC-manual/blob/main/onpolicy_distill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# On-policy distillation — Sharp CV-P09FX AC manual

**Context-distillation variant.** A small **STUDENT** (LFM2.5-350M) knows nothing about the manual. A larger frozen **TEACHER** (LFM2-1.2B) also doesn't — but if we paste the manual into the teacher's *prompt context*, it answers well. On-policy distillation moves that ability into the student's weights *without the student ever seeing the manual*:

1. **STUDENT** samples its own answer (on-policy rollout, **no** manual in context)
2. **TEACHER** scores those exact tokens, but **with** the manual in its context
3. Loss = per-token reverse KL `D_KL(student || teacher)` on the student's own tokens
4. Backprop into the student (LoRA only); teacher stays frozen

Both models are LFM2-family so they share one tokenizer (vocab 65536) → we can compare next-token distributions token-by-token. Student and teacher see **different** prompts (manual vs no-manual) but the **same** generated tokens — that mismatch is the whole point.

Runtime: **Colab → Runtime → Change runtime type → T4 GPU**. Microbatch=1 + gradient accumulation keeps memory small.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import gc, torch
# delete anything big that might still be referenced
for n in ["opt", "out", "logits", "s_logits", "t_logits", "t_logp", "s_logp", "kl", "loss"]:
    globals().pop(n, None)
gc.collect()
torch.cuda.empty_cache()
print(f"{torch.cuda.memory_allocated()/1e9:.2f} GB still allocated")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

0.00 GB still allocated
memory.used [MiB], memory.total [MiB]
0 MiB, 15360 MiB


In [3]:
# 0. Install (Colab). Safe to re-run.
!pip install -q "transformers>=4.56" "peft>=0.13" accelerate pypdf requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 8.2 MB/s eta 0:00:00


In [4]:
# 1. Imports + config
import io, re, random, requests
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

STUDENT_ID = "LiquidAI/LFM2.5-350M"   # learns; LoRA adapter trained
TEACHER_ID = "LiquidAI/LFM2-1.2B"     # frozen; gets the manual in its context

REPO_RAW = "https://raw.githubusercontent.com/kakopappa/Fine-tuning-LFM2.5-350M-on-Sharp-CV-P09FX-AC-Manual/main"
TRAIN_JSONL_URL = f"{REPO_RAW}/ac_manual_chatml_tra.jsonl"
VAL_JSONL_URL   = f"{REPO_RAW}/ac_manual_chatml_val.jsonl"
MANUAL_PDF_URL  = f"{REPO_RAW}/Sharp-CV-P09FX-AC-Manual.pdf"

SYSTEM = ("You are a helpful assistant for the Sharp CV-P09FX portable air conditioner. "
          "Answer questions accurately based on the product manual.")

# Training knobs (small on purpose -- this is for practice)
EPOCHS          = 8
MAX_NEW_TOKENS  = 128       # length of each student rollout
GEN_TEMP        = 1.0       # sampling temperature for rollouts (on-policy => sample, don't greedy)
GEN_TOP_P       = 0.95
KL_TEMP         = 1.0       # temperature used inside the KL (1.0 = compare true distributions)
LR              = 1e-4
ACCUM_STEPS     = 4         # gradient accumulation (effective batch size)
MAX_MANUAL_TOK  = 8000      # cap manual length so the teacher context stays T4-friendly
SEED            = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32   # T4 has no real bf16 -> fp16
random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE)



device: cuda


In [5]:
# 2. Data: pull the questions (we only need the user turns -- the student generates the answers)
_USER_RE = re.compile(r"<\|im_start\|>user\n(.*?)<\|im_end\|>", re.DOTALL)

def load_questions(url):
    """Each JSONL line is {\"text\": \"<ChatML conversation>\"}; extract the user question."""
    import json
    text = requests.get(url, timeout=60).text
    qs = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        m = _USER_RE.search(json.loads(line)["text"])
        if m:
            qs.append(m.group(1).strip())
    return qs

def load_manual_text(tokenizer):
    """Manual goes into the TEACHER's context. Prefer the real PDF; fall back to the training
    answers concatenated (they are manual-derived facts) if PDF extraction fails."""
    text = ""
    try:
        from pypdf import PdfReader
        raw = requests.get(MANUAL_PDF_URL, timeout=120).content
        reader = PdfReader(io.BytesIO(raw))
        text = "\n".join((p.extract_text() or "") for p in reader.pages).strip()
    except Exception as e:
        print(f"[manual] PDF extraction failed ({e}); falling back to training answers.")
    if len(text) < 200:  # extraction empty/too short -> fallback
        import json
        ans_re = re.compile(r"<\|im_start\|>assistant\n(.*?)<\|im_end\|>", re.DOTALL)
        lines = requests.get(TRAIN_JSONL_URL, timeout=60).text.splitlines()
        facts = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            m = ans_re.search(json.loads(line)["text"])
            if m:
                facts.append("- " + m.group(1).strip())
        text = "Key facts from the Sharp CV-P09FX manual:\n" + "\n".join(facts)
    ids = tokenizer(text, add_special_tokens=False)["input_ids"][:MAX_MANUAL_TOK]
    return tokenizer.decode(ids)

In [6]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [7]:
# 3. Tokenizer + models
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Safety check: token-level KL is only valid if both models share the same vocab mapping.
t_tok = AutoTokenizer.from_pretrained(TEACHER_ID)
probe = "Window panel width 22 inches (559mm). Drain the condensate."
assert tokenizer(probe)["input_ids"] == t_tok(probe)["input_ids"], \
    "Student and teacher tokenizers disagree -- token-level KL would be meaningless."
del t_tok
print("Tokenizer alignment OK.")

teacher = AutoModelForCausalLM.from_pretrained(TEACHER_ID, torch_dtype=DTYPE, attn_implementation="sdpa").to(DEVICE).eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student = AutoModelForCausalLM.from_pretrained(STUDENT_ID, torch_dtype=DTYPE).to(DEVICE)
student = get_peft_model(student, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],  # LFM2 attention projections
    task_type="CAUSAL_LM",
))
student.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

trainable params: 2,031,616 || all params: 356,515,584 || trainable%: 0.5699


In [8]:
# 4. Prompt builders + rollout. Student context has NO manual; teacher context HAS the manual.
def build_prompt_ids(question, with_manual, manual_text):
    sys_content = SYSTEM
    if with_manual:
        sys_content = SYSTEM + "\n\nProduct manual:\n" + manual_text
    messages = [{"role": "system", "content": sys_content},
                {"role": "user", "content": question}]
    # return_dict=True is explicit so we reliably get a tensor out of ["input_ids"] (recent
    # transformers return a BatchEncoding from apply_chat_template, which has no .shape).
    ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True)["input_ids"]
    return ids.to(DEVICE)

@torch.no_grad()
def student_rollout(question):
    """Sample one on-policy answer from the student (no manual). Returns (student_prompt, gen_ids)."""
    prompt = build_prompt_ids(question, with_manual=False, manual_text=None)
    out = student.generate(
        prompt, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
        temperature=GEN_TEMP, top_p=GEN_TOP_P, pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[:, prompt.shape[1]:]                 # only the newly generated tokens
    eos = tokenizer.eos_token_id                   # trim after first EOS so we don't distill padding
    if eos is not None and (gen[0] == eos).any():
        gen = gen[:, : (gen[0] == eos).nonzero()[0, 0].item() + 1]
    return prompt, gen

def logits_on_gen(model, prompt_ids, gen_ids):
    """Run model on [prompt ++ gen]; return logits PREDICTING each gen token: [gen_len, V].
    Logit at position p predicts token p+1, so the gen-predicting logits start at prompt_len-1."""
    full = torch.cat([prompt_ids, gen_ids], dim=1)
    logits = model(full).logits[0]                 # [seq, V]
    start = prompt_ids.shape[1] - 1
    return logits[start : start + gen_ids.shape[1]]  # [gen_len, V]

In [9]:
# 5. The on-policy distillation step
def distill_loss(question, manual_text):
    prompt_s, gen = student_rollout(question)
    if gen.shape[1] == 0:
        return None
    # Teacher scores the SAME gen tokens, but from a manual-rich context (frozen, no grad).
    prompt_t = build_prompt_ids(question, with_manual=True, manual_text=manual_text)
    with torch.no_grad():
        t_logits = logits_on_gen(teacher, prompt_t, gen).float()
    # Student re-scores its own gen tokens WITH grad (generate() produced no graph).
    s_logits = logits_on_gen(student, prompt_s, gen).float()
    # Per-token reverse KL  D_KL(student || teacher) = sum_v p_s (log p_s - log p_t).
    s_logp = F.log_softmax(s_logits / KL_TEMP, dim=-1)
    t_logp = F.log_softmax(t_logits / KL_TEMP, dim=-1)
    kl = (s_logp.exp() * (s_logp - t_logp)).sum(dim=-1)   # [gen_len]
    return kl.mean()

In [10]:
# 6. Eval helpers: a greedy answer (qualitative) and held-out mean reverse-KL (quantitative)
@torch.no_grad()
def answer(question):
    prompt = build_prompt_ids(question, with_manual=False, manual_text=None)
    out = student.generate(prompt, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, prompt.shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def eval_kl(questions, manual_text):
    """Mean per-token reverse KL on held-out questions, using GREEDY rollouts so the number is
    deterministic and comparable before vs after. Lower = student closer to the manual-informed
    teacher = more knowledge absorbed."""
    was_training = student.training
    student.eval()
    total_kl = 0.0; total_tok = 0
    for q in questions:
        prompt_s = build_prompt_ids(q, with_manual=False, manual_text=None)
        out = student.generate(prompt_s, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                               pad_token_id=tokenizer.pad_token_id)
        gen = out[:, prompt_s.shape[1]:]
        eos = tokenizer.eos_token_id
        if eos is not None and (gen[0] == eos).any():
            gen = gen[:, : (gen[0] == eos).nonzero()[0, 0].item() + 1]
        if gen.shape[1] == 0:
            continue
        prompt_t = build_prompt_ids(q, with_manual=True, manual_text=manual_text)
        s_logp = F.log_softmax(logits_on_gen(student, prompt_s, gen).float() / KL_TEMP, dim=-1)
        t_logp = F.log_softmax(logits_on_gen(teacher, prompt_t, gen).float() / KL_TEMP, dim=-1)
        kl = (s_logp.exp() * (s_logp - t_logp)).sum(dim=-1)
        total_kl += kl.sum().item(); total_tok += kl.shape[0]
    if was_training:
        student.train()
    return total_kl / max(total_tok, 1)

In [11]:
def build_relevant_context(tokenizer, cap=2500):
    from pypdf import PdfReader
    raw = requests.get(MANUAL_PDF_URL, timeout=120).content
    text = "\n".join((p.extract_text() or "") for p in PdfReader(io.BytesIO(raw)).pages)
    kw = ("inch","mm","width","window","screw","drain","panel","minimum","maximum",
          "559","installation","filter","temperature","btu","watt","volt","hose","exhaust")
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    kept  = [ln for ln in lines if any(k in ln.lower() for k in kw)]
    ctx = "Relevant specs from the Sharp CV-P09FX manual:\n" + "\n".join(kept)
    ids = tokenizer(ctx, add_special_tokens=False)["input_ids"][:cap]
    return tokenizer.decode(ids)

questions  = load_questions(TRAIN_JSONL_URL)
val_qs     = load_questions(VAL_JSONL_URL)
manual     = build_relevant_context(tokenizer)
kl_eval_qs = val_qs[:20]
print("manual tokens:", len(tokenizer(manual)['input_ids']), "| '559':", "559" in manual)

manual tokens: 2501 | '559': True


## Before training
Baseline: the untrained student's held-out reverse-KL to the teacher, plus a few sample answers.

In [12]:
print(f"held-out mean reverse-KL: {eval_kl(kl_eval_qs, manual):.4f}")
for q in val_qs[:3]:
    print(f"Q: {q}\nA: {answer(q)}\n")

held-out mean reverse-KL: 1.1133
Q: What is the minimum window width for the window panel?
A: The minimum window width for the window panel is 1.5 inches.

Q: What should I do if my window width is between 22 and 24 inches?
A: If your window width is between 22 and 24 inches, you should consider upgrading to a larger window size. This will improve airflow and reduce heat gain, making your home more comfortable during hot weather.

Q: How many screws do I use for a 30-inch wide window?
A: I don't have access to specific product manuals or technical specifications. To get accurate information about the number of screws for a 30-inch wide window, I recommend checking the product manual or contacting the manufacturer directly.



## Train
Watch the running `reverseKL` drop. (Re-run this cell to train more epochs.)

In [13]:
opt = torch.optim.AdamW([p for p in student.parameters() if p.requires_grad], lr=LR)
student.train()

step = 0
for epoch in range(EPOCHS):
    random.shuffle(questions)
    running = 0.0; counted = 0
    opt.zero_grad()
    for i, q in enumerate(questions):
        loss = distill_loss(q, manual)
        if loss is None:
            continue
        (loss / ACCUM_STEPS).backward()
        running += loss.item(); counted += 1
        if (i + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in student.parameters() if p.requires_grad], 1.0)
            opt.step(); opt.zero_grad(); step += 1
            if step % 5 == 0:
                print(f"epoch {epoch} step {step}  reverseKL={running/max(counted,1):.4f}")
                running = 0.0; counted = 0
    opt.step(); opt.zero_grad()  # flush remainder
print("done.")

epoch 0 step 5  reverseKL=0.9933
epoch 0 step 10  reverseKL=0.6739
epoch 1 step 15  reverseKL=0.5735
epoch 1 step 20  reverseKL=0.5644
epoch 2 step 25  reverseKL=0.5532
epoch 2 step 30  reverseKL=0.5759
epoch 3 step 35  reverseKL=0.5320
epoch 3 step 40  reverseKL=0.5196
epoch 4 step 45  reverseKL=0.4755
epoch 4 step 50  reverseKL=0.4953
epoch 4 step 55  reverseKL=0.5579
epoch 5 step 60  reverseKL=0.5206
epoch 5 step 65  reverseKL=0.4836
epoch 6 step 70  reverseKL=0.5333
epoch 6 step 75  reverseKL=0.5284
epoch 7 step 80  reverseKL=0.4588
epoch 7 step 85  reverseKL=0.5130
done.


## After training
Reverse-KL should be lower than the BEFORE number, and the answers more manual-accurate.

In [14]:
student.eval()
print(f"held-out mean reverse-KL: {eval_kl(kl_eval_qs, manual):.4f}")
for q in val_qs[:3]:
    print(f"Q: {q}\nA: {answer(q)}\n")

out_dir = "./lfm25_350m_ac_onpolicy_lora"
student.save_pretrained(out_dir)
print(f"Saved student LoRA adapter to {out_dir}")

held-out mean reverse-KL: 0.5633
Q: What is the minimum window width for the window panel?
A: The minimum window width for the window panel is not explicitly stated in the provided manual. However, it is generally recommended that the window panel has a minimum width of at least 1 inch (2.5 cm) to ensure proper airflow and prevent the window from becoming too narrow, which could lead to issues with air circulation and potential damage to the window.

For a standard window panel, the minimum window width is typically around 1 inch (2.5 cm) in width. However, this can vary depending on the specific model and manufacturer's specifications. It's always best to refer to the manufacturer's guidelines or the product manual

Q: What should I do if my window width is between 22 and 24 inches?
A: If your window width is between 22 and 24 inches, you should consider the following steps to ensure your air conditioner operates efficiently:

1. **Check the Window Size**: Ensure that your window widt

In [15]:
q = "What is the minimum window width for the window panel?"
pt = build_prompt_ids(q, with_manual=True, manual_text=manual)
out = teacher.generate(pt, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.pad_token_id)
print(tokenizer.decode(out[0, pt.shape[1]:], skip_special_tokens=True).strip())

The minimum window width for the window panel is between 22 inches (559mm) and 24 inches (609mm). If the inner width of the window is within this range, the window panel can be installed. However, if the window width is less than 22 inches (559mm), the window panel cannot be installed because it would be unable to shut the exhaust cover properly.


In [16]:
# Diagnostic: is the manual really in context, and does the TEACHER answer correctly?
print("manual tokens:", len(tokenizer(manual)['input_ids']))
print("contains '22'? ", "22" in manual, " | contains '559'? ", "559" in manual)
print("manual head:\n", manual[:400], "\n---\n")

for q in val_qs[:3]:
    pt = build_prompt_ids(q, with_manual=True, manual_text=manual)
    out = teacher.generate(pt, max_new_tokens=128, do_sample=False,
                           pad_token_id=tokenizer.pad_token_id)
    print("Q:", q)
    print("TEACHER+manual:", tokenizer.decode(out[0, pt.shape[1]:], skip_special_tokens=True).strip(), "\n")

manual tokens: 2501
contains '22'?  True  | contains '559'?  True
manual head:
 Relevant specs from the Sharp CV-P09FX manual:
INSTALLATION AND OPERATION
MANUEL D’INSTALLATION ET
commerce de SHARP Corporation.
•INSTALL WINDOW PANEL ................................... E-12
•INSTALLATION AND REMOVAL OF EXHAUST HOSE... E-16
•DRAINAGE ............................................................ E-29
voltage or other misuse, abnormal service or handling, or which has been altered  
---

Q: What is the minimum window width for the window panel?
TEACHER+manual: The minimum window width for the window panel is between 22 inches (559mm) and 24 inches (609mm). If the inner width of the window is within this range, the window panel can be installed. However, if the window width is less than 22 inches (559mm), the window panel cannot be installed because it would be unable to shut the exhaust cover properly. 

Q: What should I do if my window width is between 22 and 24 inches?
TEACHER+manual: If y

In [17]:
# (A) What does the teacher actually SEE about width? (is the spec a coherent line?)
print("--- context lines mentioning width/numbers ---")
for l in manual.splitlines():
    if any(k in l.lower() for k in ["width","inch","559","minimum"]):
        print(repr(l))

# (B) What does the teacher ANSWER with this context? (greedy)
print("\n--- teacher answers ---")
for q in ["What is the minimum window width for the window panel?",
          "What should I do if my window width is between 22 and 24 inches?"]:
    pt = build_prompt_ids(q, with_manual=True, manual_text=manual)
    out = teacher.generate(pt, max_new_tokens=120, do_sample=False,
                           pad_token_id=tokenizer.pad_token_id)
    print("Q:", q)
    print("TEACHER:", tokenizer.decode(out[0, pt.shape[1]:], skip_special_tokens=True).strip(), "\n")

--- context lines mentioning width/numbers ---
'If the inner width of the window is be-'
'tween 22" (559mm) and 24" (609mm)'
'windows less than 22" (559mm) wide, as'
'panel to the same width as the window.'
'If the inner width of the window is'
'window frame width.'
'If the inner width of the window is'
'panels to fit the window frame width.'
'between 22" (559mm) and 24" (609mm)'
'windows less than 22" (559mm) high, as'
'DEHUMIDIFICATION with container(minimum capacity 3 gallons) Remove'
'• Insert the hose securely into a container with a minimum'

--- teacher answers ---
Q: What is the minimum window width for the window panel?
TEACHER: The minimum window width for the window panel is between 22 inches (559mm) and 24 inches (609mm). If the inner width of the window is within this range, the window panel can be installed. However, if the window width is less than 22 inches (559mm), the window panel cannot be installed because it would be unable to shut the exhaust cover properly. 

Q: 